In [1]:
import numpy as np

## Part I

Дан массив A размера (X, Y, Z) -- к примеру, (1000, 1000, 30). Отдельный элемент A[i, j, :] назовем трассой из A.
дан второй массив B размера (N, Z) -- несколько трасс. Необходимо для каждой трассы из A посчитать среднее значение корреляции между A[i, j, :] и всеми трассами из B. Примерное значение N -- от 20 до 100


### Solution via for loop (slow)

In [2]:
def exchange_correlation_slow(A, B):
    
    X, Y, Z = A.shape # 1000 x 1000 x 30
    mean_corr_for_traces = np.empty((X, Y))
    
    for i in range(X):
        for j in range(Y):
            res = np.mean((A[i, j] - A[i, j].mean()) * (B - B.mean(axis=-1, keepdims=True)), axis=-1) /  (A[i, j].std() * B.std(axis=-1))
            mean_corr_for_traces[i, j] = np.mean(res)
    
    return mean_corr_for_traces

Some notes

In [3]:
A = np.random.randn(1000, 1000, 30)

In [4]:
%%timeit
u = np.zeros_like(A)

52.2 ms ± 224 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
u = np.zeros((A.shape[0], A.shape[1]))

295 µs ± 473 ns per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [6]:
%%timeit
u = np.empty_like(A)

5.59 µs ± 318 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [7]:
%%timeit
u = np.empty((A.shape[0], A.shape[1]))

594 ns ± 4.51 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [8]:
x = np.random.randn(10000)
y = np.random.randn(10000) 

In [9]:
%%time
x @ y 

CPU times: user 39 µs, sys: 47 µs, total: 86 µs
Wall time: 103 µs


25.249357953908998

In [10]:
%%time
(x * y).sum()

CPU times: user 257 µs, sys: 0 ns, total: 257 µs
Wall time: 203 µs


25.249357953908948

### Solution via matrix mul (fast)

In [11]:
def exchange_correlation(A, B):
    """ For each trace in the trace matrix A computes mean correlation with each trace in trace matrix B
    
    Parameters 
    ----------
    A: np.ndarray, shape = (X, Y, Z)
        First trace matrix. Matrix A has X * Y traces.
    B: np.ndarray, shape = (N, Z)
        Second trace matrix.
        
    Returns
    -------
    correlation_matrix: np.ndarray, shape = (X, Y)
        Matrix with mean values of correlation for each trace in A. 
    """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    A_reshaped = A.reshape(X * Y, -1)
    
    # normalize matrices to compute correaltions
    A_normalized = (A_reshaped - A_reshaped.mean(axis=-1, keepdims=True)) / A_reshaped.std(axis=-1, keepdims=True)
    B_normalized = (B - B.mean(axis=-1, keepdims=True)) / B.std(axis=-1, keepdims=True)
    
    # compute correlation for each trace in A matrix (we have X * Y traces)
    correlation_matrix = np.mean(1 / Z * A_normalized @ B_normalized.T, axis=-1).reshape(X, -1) # shape = (X, Y)
    
    return correlation_matrix

### Trials

In [12]:
A = np.random.randn(1000, 1000, 30)
#A = np.random.randn(4, 4, 5)

### Let's compare slow and fast solution with N = 20

In [13]:
N = 20
B = np.random.randn(N, 30)

In [14]:
%%time 
output_slow_1 = exchange_correlation_slow(A, B)
print(output_slow_1.shape)
output_slow_1

(1000, 1000)
CPU times: user 1min 9s, sys: 12.2 ms, total: 1min 9s
Wall time: 1min 9s


array([[-0.03554713, -0.03608682,  0.02618619, ...,  0.02420111,
         0.01002518,  0.01805935],
       [ 0.00034792, -0.02211397, -0.02542373, ...,  0.07880729,
         0.00530402, -0.0514903 ],
       [-0.01059165,  0.00268263, -0.06206702, ..., -0.02524028,
        -0.08690624, -0.03478253],
       ...,
       [-0.04727238,  0.0083004 , -0.00569354, ..., -0.03422653,
         0.03946547,  0.01498157],
       [-0.01522549, -0.04111209, -0.02216511, ..., -0.02941312,
        -0.03745683, -0.02249895],
       [-0.01251368,  0.05836485, -0.01941487, ...,  0.10556256,
         0.0080297 ,  0.03135104]])

In [15]:
%%time
output_fast_1 = exchange_correlation(A, B)
print(output_fast_1.shape)
output_fast_1

(1000, 1000)
CPU times: user 3.18 s, sys: 3.27 s, total: 6.45 s
Wall time: 505 ms


array([[-0.03554713, -0.03608682,  0.02618619, ...,  0.02420111,
         0.01002518,  0.01805935],
       [ 0.00034792, -0.02211397, -0.02542373, ...,  0.07880729,
         0.00530402, -0.0514903 ],
       [-0.01059165,  0.00268263, -0.06206702, ..., -0.02524028,
        -0.08690624, -0.03478253],
       ...,
       [-0.04727238,  0.0083004 , -0.00569354, ..., -0.03422653,
         0.03946547,  0.01498157],
       [-0.01522549, -0.04111209, -0.02216511, ..., -0.02941312,
        -0.03745683, -0.02249895],
       [-0.01251368,  0.05836485, -0.01941487, ...,  0.10556256,
         0.0080297 ,  0.03135104]])

In [16]:
np.allclose(output_slow_1, output_fast_1)

True

### Fast solution with different N

In [17]:
%%time
N = 50 
B = np.random.randn(N, 30)
output = exchange_correlation(A, B)
print(output.shape)
output

(1000, 1000)
CPU times: user 8.4 s, sys: 6.79 s, total: 15.2 s
Wall time: 621 ms


array([[-0.02128441, -0.04158472, -0.03196858, ..., -0.01181892,
        -0.03089459,  0.0146152 ],
       [-0.00787602, -0.00736769, -0.00625628, ...,  0.04985168,
         0.03125166, -0.00531116],
       [-0.02621923,  0.0123293 ,  0.06094897, ..., -0.00568086,
        -0.06407641,  0.00620572],
       ...,
       [-0.08337441, -0.03208295,  0.00259202, ...,  0.03167413,
         0.0023188 ,  0.05431179],
       [ 0.00518689, -0.0046685 , -0.00714367, ..., -0.06018637,
        -0.0110242 ,  0.01038029],
       [ 0.03958794, -0.04769527, -0.00274511, ...,  0.02588254,
        -0.0076513 ,  0.00732088]])

In [18]:
%%time
N = 70 
B = np.random.randn(N, 30)
output = exchange_correlation(A, B)
print(output.shape)
output

(1000, 1000)
CPU times: user 9.96 s, sys: 9.09 s, total: 19 s
Wall time: 694 ms


array([[-0.00363377,  0.00448067, -0.03311618, ...,  0.03068776,
        -0.00167162, -0.00696628],
       [ 0.02786946,  0.01200378,  0.04832472, ...,  0.00368178,
         0.00392237,  0.00106218],
       [ 0.02724776, -0.00748955, -0.01426725, ..., -0.01986557,
         0.02068023, -0.00781805],
       ...,
       [ 0.01885863,  0.00110159,  0.01050195, ..., -0.02338045,
         0.01144555, -0.02047683],
       [ 0.02426249, -0.00283034, -0.01030485, ...,  0.00729711,
         0.01329256,  0.00652544],
       [-0.00138606, -0.02516676, -0.01774535, ...,  0.00093823,
         0.03279617, -0.00862967]])

In [19]:
%%time
N = 100 
B = np.random.randn(N, 30)
output = exchange_correlation(A, B)
print(output.shape)
output

(1000, 1000)
CPU times: user 11.1 s, sys: 9.96 s, total: 21 s
Wall time: 762 ms


array([[ 2.31206014e-03, -2.50204351e-02,  1.30622510e-04, ...,
        -4.81490748e-03, -5.05911502e-03, -5.54885908e-05],
       [ 7.84628784e-03, -1.12438846e-02,  1.75898839e-02, ...,
         4.99275231e-03, -2.58700934e-02,  3.76622796e-03],
       [ 2.40569486e-03, -1.74082976e-03, -2.30869670e-02, ...,
        -2.07206381e-03,  2.78156210e-02,  1.02256354e-03],
       ...,
       [-1.63240224e-02, -1.93757024e-02, -3.70167414e-03, ...,
        -1.12942948e-02, -1.36461577e-02, -7.83325272e-03],
       [ 2.44394792e-02,  1.41861419e-02, -4.61644111e-02, ...,
        -3.38343152e-03, -1.97620946e-02,  3.76855638e-03],
       [ 1.12614877e-02,  3.64275906e-03,  7.55722157e-03, ...,
         2.72878970e-02,  1.18419843e-02, -2.41152743e-02]])

## Part II

Для квадратного окна заданного (нечетного) размера K посчитать массив output размера (X, Y), где на i,j месте находится среднее значение корреляции между трассой A[i, j, :] и всеми соседними трассами, лежащими внутри квадратного окна KxK с центром в i,j трассе. 
Примерное значение K -- от 3 до 13

### Slow solution 

In [20]:
def window_correlation(A, K):
    """ Applies sliding window to i, j trace in trace matrix A via np.corrcoef. """
    X, Y = A.shape[:2]
    output = np.empty((X, Y))
    
    for i in range(X):
        for j in range(Y):
            
            if i == 0 or i == A.shape[0] - 1:
                K_x = K - 1
            else: 
                K_x = K
                
            if j == 0 or j == A.shape[1] - 1:
                K_y = K - 1
            else:
                K_y = K
                
            sum_corr = 0
            values_in_window = 0
            start_idx_j = max(0, j - K_y // 2)
            final_idx_j = min(Y, j + K_y // 2 + 1)
            start_idx_i = max(0, i - K_x // 2)
            final_idx_i = min(X, i + K_x // 2 + 1) 
            
            for window_i in range(start_idx_i, final_idx_i):
                for window_j in range(start_idx_j, final_idx_j):
                    
                    if i == window_i and j == window_j: # don't consider the central trace with itself 
                        continue
                        
                    sum_corr += np.corrcoef(A[i, j], A[window_i, window_j])[0, 1]
                    values_in_window += 1
                    
            avg_corr = sum_corr / values_in_window
            
            output[i, j] = avg_corr
    
    return output 

In [21]:
def window_correlation_handy_corr(A, K):
    """ Applies sliding window to i, j trace in trace matrix A via handy correlations. """
    X, Y = A.shape[:2]
    output = np.empty((X, Y))
    
    for i in range(X):
        for j in range(Y):
            
            if i == 0 or i == A.shape[0] - 1:
                K_x = K - 1
            else: 
                K_x = K
                
            if j == 0 or j == A.shape[1] - 1:
                K_y = K - 1
            else:
                K_y = K
                
            sum_corr = 0
            values_in_window = 0
            start_idx_j = max(0, j - K_y // 2)
            final_idx_j = min(Y, j + K_y // 2 + 1)
            start_idx_i = max(0, i - K_x // 2)
            final_idx_i = min(X, i + K_x // 2 + 1) 
            
            for window_i in range(start_idx_i, final_idx_i):
                for window_j in range(start_idx_j, final_idx_j):
                    
                    if i == window_i and j == window_j: # don't consider the central trace with itself 
                        continue
                        
                    sum_corr += np.mean((A[i, j] - A[i, j].mean()) * (A[window_i, window_j] - A[window_i, window_j].mean())) \
                                / (A[i, j].std() * A[window_i, window_j].std())
                    values_in_window += 1
                    
            avg_corr = sum_corr / values_in_window
            
            output[i, j] = avg_corr
    
    return output 

### Fast solution

In [22]:
def trace_correlation(A, K):
    """ For window KxK computes mean correlation for trace in the middle of the window and its neighbours. 
    
    Parameters
    ----------
    A: np.ndarray, shape = (X, Y, Z)
        Trace matrix
    K: int, odd number
        Size of the kernel 
        
    Returns 
    -------
    mean_corrs: np.ndarray, shape = (X, Y)
       Matrix with mean correlation for each trace in the i, j position   
    """
    # other parameters
    padding = K // 2 
    mid = K ** 2 // 2 # idx of mid trace 
    X, Y, Z = A.shape # 1000 x 1000 x 30 
    
    # add padding to the trace matrix 
    A_pad = np.pad(A, pad_width=((padding, padding), (padding, padding), (0, 0)))
    
    # make K x K x Z patches
    patches = np.lib.stride_tricks.sliding_window_view(A_pad, window_shape=(K, K, Z)) # shape = (X, Y, 1, K, K, Z)
    patches_reshaped = patches.reshape(patches.shape[0] * patches.shape[1],
                                       patches.shape[2] * patches.shape[3] * patches.shape[4],
                                       patches.shape[5]) # shape = (X * Y, K * K, Z)
    
    # normalize and transpose patches to compute correlations
    patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \
                    / patches_reshaped.std(axis=-1, keepdims=True) 
    patches_norm_transposed = patches_norm.transpose((0, 2, 1))
    
    # find correlations for each trace in each patch and choose only mid trace
    cor_matrix = 1 / Z * (patches_norm @ patches_norm_transposed)[:, mid, :]
    cor_matrix[:, mid] = np.nan # replace ones to nans so that don't consider trace with itslef  
    mean_corrs = np.nanmean(cor_matrix, axis=-1).reshape(X, -1) # compute average correlation for each trace
    
    return mean_corrs

### Trials

In [23]:
A = np.random.randn(1000, 1000, 30)

### Let's compare two algorithms on K = 3 and K = 5 

In [24]:
%%time
K = 3
output_slow_1 = window_correlation(A, K)
print(output_slow_1.shape)
output_slow_1

(1000, 1000)
CPU times: user 8min 29s, sys: 69.3 ms, total: 8min 29s
Wall time: 8min 29s


array([[ 0.02877342,  0.01461295,  0.07961322, ...,  0.0781015 ,
         0.02059396,  0.14089492],
       [-0.04616099, -0.03699804, -0.10900431, ..., -0.09259367,
        -0.08974376, -0.17259217],
       [-0.02719416, -0.04342897,  0.01009337, ...,  0.06758437,
         0.06345471,  0.01868396],
       ...,
       [-0.05216101, -0.02814209,  0.01465085, ...,  0.02964628,
        -0.04750574,  0.08530123],
       [ 0.06004572, -0.09460723, -0.0582226 , ...,  0.00240383,
        -0.01059433,  0.07179911],
       [-0.05287317,  0.00713992,  0.06315951, ...,  0.02714994,
         0.00582981, -0.04510729]])

In [25]:
%%time
K = 3
output_slow_handy_1 = window_correlation_handy_corr(A, K)
print(output_slow_handy_1.shape)
output_slow_handy_1

(1000, 1000)
CPU times: user 6min 35s, sys: 15.2 ms, total: 6min 35s
Wall time: 6min 35s


array([[ 0.02877342,  0.01461295,  0.07961322, ...,  0.0781015 ,
         0.02059396,  0.14089492],
       [-0.04616099, -0.03699804, -0.10900431, ..., -0.09259367,
        -0.08974376, -0.17259217],
       [-0.02719416, -0.04342897,  0.01009337, ...,  0.06758437,
         0.06345471,  0.01868396],
       ...,
       [-0.05216101, -0.02814209,  0.01465085, ...,  0.02964628,
        -0.04750574,  0.08530123],
       [ 0.06004572, -0.09460723, -0.0582226 , ...,  0.00240383,
        -0.01059433,  0.07179911],
       [-0.05287317,  0.00713992,  0.06315951, ...,  0.02714994,
         0.00582981, -0.04510729]])

In [26]:
%%time
K = 3
output_fast_1 = trace_correlation(A=A, K=K)
print(output_fast_1.shape)
output_fast_1

/tmp/ipykernel_24630/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 2.8 s, sys: 1.85 s, total: 4.65 s
Wall time: 4.64 s


array([[ 0.02877342,  0.01461295,  0.07961322, ...,  0.0781015 ,
         0.02059396,  0.14089492],
       [-0.04616099, -0.03699804, -0.10900431, ..., -0.09259367,
        -0.08974376, -0.17259217],
       [-0.02719416, -0.04342897,  0.01009337, ...,  0.06758437,
         0.06345471,  0.01868396],
       ...,
       [-0.05216101, -0.02814209,  0.01465085, ...,  0.02964628,
        -0.04750574,  0.08530123],
       [ 0.06004572, -0.09460723, -0.0582226 , ...,  0.00240383,
        -0.01059433,  0.07179911],
       [-0.05287317,  0.00713992,  0.06315951, ...,  0.02714994,
         0.00582981, -0.04510729]])

In [27]:
np.allclose(output_slow_1, output_slow_handy_1), np.allclose(output_slow_1, output_fast_1)

(True, True)

In [28]:
%%time
K = 5
output_slow_2 = window_correlation(A, K)
print(output_slow_2.shape)
output_slow_2

(1000, 1000)
CPU times: user 25min 22s, sys: 61.1 ms, total: 25min 22s
Wall time: 25min 22s


array([[ 0.01882032, -0.00368158, -0.00589589, ..., -0.02655963,
         0.02828856,  0.02692485],
       [-0.02420783, -0.07642751, -0.0547327 , ...,  0.0106696 ,
        -0.0809791 , -0.07339894],
       [ 0.03805674, -0.05209683, -0.01471849, ...,  0.01167707,
         0.03483807,  0.12562046],
       ...,
       [ 0.00998142, -0.0309503 ,  0.06931256, ..., -0.00097961,
        -0.04146055,  0.06819835],
       [ 0.02785992, -0.02054714, -0.01735164, ...,  0.01618399,
        -0.00546443,  0.03823029],
       [-0.03748044, -0.06307286,  0.0008934 , ...,  0.00957601,
        -0.06389048, -0.05636094]])

In [29]:
%%time
K = 5
output_slow_handy_2 = window_correlation(A, K)
print(output_slow_handy_2.shape)
output_slow_handy_2

(1000, 1000)
CPU times: user 25min 48s, sys: 196 ms, total: 25min 48s
Wall time: 25min 49s


array([[ 0.01882032, -0.00368158, -0.00589589, ..., -0.02655963,
         0.02828856,  0.02692485],
       [-0.02420783, -0.07642751, -0.0547327 , ...,  0.0106696 ,
        -0.0809791 , -0.07339894],
       [ 0.03805674, -0.05209683, -0.01471849, ...,  0.01167707,
         0.03483807,  0.12562046],
       ...,
       [ 0.00998142, -0.0309503 ,  0.06931256, ..., -0.00097961,
        -0.04146055,  0.06819835],
       [ 0.02785992, -0.02054714, -0.01735164, ...,  0.01618399,
        -0.00546443,  0.03823029],
       [-0.03748044, -0.06307286,  0.0008934 , ...,  0.00957601,
        -0.06389048, -0.05636094]])

In [30]:
%%time
K = 5
output_fast_2 = trace_correlation(A=A, K=K)
print(output_fast_2.shape)
output_fast_2

/tmp/ipykernel_24630/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 8.35 s, sys: 5.25 s, total: 13.6 s
Wall time: 13.6 s


array([[ 0.01882032, -0.00368158, -0.00589589, ..., -0.02655963,
         0.02828856,  0.02692485],
       [-0.02420783, -0.07642751, -0.0547327 , ...,  0.0106696 ,
        -0.0809791 , -0.07339894],
       [ 0.03805674, -0.05209683, -0.01471849, ...,  0.01167707,
         0.03483807,  0.12562046],
       ...,
       [ 0.00998142, -0.0309503 ,  0.06931256, ..., -0.00097961,
        -0.04146055,  0.06819835],
       [ 0.02785992, -0.02054714, -0.01735164, ...,  0.01618399,
        -0.00546443,  0.03823029],
       [-0.03748044, -0.06307286,  0.0008934 , ...,  0.00957601,
        -0.06389048, -0.05636094]])

In [31]:
np.allclose(output_slow_2, output_slow_handy_2), np.allclose(output_slow_2, output_fast_2)

(True, True)

### Fast solution on K from 7 to 13

In [32]:
%%time
K = 7
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_24630/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 17.8 s, sys: 11.6 s, total: 29.4 s
Wall time: 29.4 s


array([[ 0.07020237,  0.07336611, -0.02782333, ..., -0.05013462,
        -0.01429768, -0.02487518],
       [-0.02083194, -0.08027214, -0.01789075, ...,  0.02935251,
        -0.09721096,  0.00121655],
       [-0.00476456, -0.06633906, -0.00924851, ...,  0.03236249,
        -0.00324245,  0.06521739],
       ...,
       [-0.01544891, -0.01183882,  0.00903582, ...,  0.00013266,
        -0.03610156,  0.07558866],
       [ 0.04976057, -0.0358852 , -0.01937085, ..., -0.00607936,
        -0.01423013,  0.059254  ],
       [-0.05102617, -0.00612535, -0.01858676, ...,  0.0609021 ,
         0.00549586, -0.00616312]])

In [33]:
%%time
K = 9
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_24630/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 31.3 s, sys: 23.2 s, total: 54.5 s
Wall time: 54.5 s


array([[ 0.02758535,  0.00533692,  0.00620106, ..., -0.04073039,
        -0.02474217, -0.01640702],
       [-0.02822983, -0.06170024,  0.01225561, ...,  0.03134333,
        -0.06150284,  0.00673622],
       [ 0.02751162, -0.0577141 , -0.0340105 , ...,  0.03413909,
         0.01043313,  0.0270232 ],
       ...,
       [ 0.02164541, -0.03881338,  0.013006  , ...,  0.00584934,
        -0.02992071,  0.06545061],
       [ 0.00581924, -0.01762504, -0.00144987, ...,  0.00800151,
        -0.00271758,  0.04925118],
       [ 0.01428099, -0.02114784, -0.01522604, ...,  0.05589628,
        -0.02402486, -0.02834371]])

In [34]:
%%time
K = 11
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_24630/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 50.1 s, sys: 42.1 s, total: 1min 32s
Wall time: 1min 32s


array([[-0.0006126 ,  0.03058487,  0.00840657, ..., -0.01022735,
        -0.0288375 , -0.01119481],
       [-0.02494518, -0.03657165,  0.00217539, ...,  0.02953726,
        -0.03440721,  0.00192782],
       [ 0.01136941, -0.06892647, -0.00771819, ...,  0.03684024,
         0.02513474,  0.02705453],
       ...,
       [ 0.00020625, -0.03926661, -0.00266509, ..., -0.00893305,
        -0.01367422,  0.04424009],
       [ 0.00480494, -0.02480264, -0.0083434 , ...,  0.00063933,
        -0.01278261,  0.01936832],
       [-0.01100048, -0.0184208 , -0.03318417, ...,  0.03574164,
        -0.02402595,  0.00525166]])

In [35]:
%%time
K = 13
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_24630/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 5min 17s, sys: 11min 8s, total: 16min 26s
Wall time: 3min 42s


array([[ 0.00364566,  0.01817287,  0.01212385, ..., -0.0169815 ,
        -0.02998546, -0.00782242],
       [ 0.00203734, -0.03064113, -0.00716536, ...,  0.01557918,
        -0.04091706,  0.01703157],
       [ 0.01182066, -0.04503445,  0.00127124, ...,  0.03582273,
         0.02880638,  0.03788748],
       ...,
       [-0.0081715 , -0.03389134, -0.00624686, ...,  0.00173138,
        -0.00104326,  0.04646457],
       [ 0.01423347, -0.03153361, -0.00587118, ...,  0.00988505,
        -0.01470693, -0.00385002],
       [ 0.02791944, -0.02356385, -0.01783615, ...,  0.03238638,
        -0.01133452,  0.00787669]])